# Web Scraping and Data Cleaning of Tour de France Rider Histor

## Introduction

This notebook covers the data cleaning and export phase of the Tour de France Data Science Project.

Our primary data source is [Wikipedia’s list of Tour de France general classification winners](https://en.wikipedia.org/wiki/List_of_Tour_de_France_general_classification_winners), which provides comprehensive information on every champion since 1903.

We will use web scraping to extract the raw HTML tables, clean and restructure the critical data, and export everything into reproducible formats (**CSV**, **JSON**, and **Pickle**). These cleaned datasets lay the groundwork for all subsequent analysis and visualization.


## Focus of This Notebook

- Ensuring the data is consistent, readable, and analysis-ready  
- Making every step transparent and well-documented for easy re-use by others

## Libraries and Tools Used

The following Python libraries will be used and should be installed before running the notebook:

- **requests**: HTTP requests to fetch web pages  
- **BeautifulSoup**: Parse HTML and extract the needed content  
- **csv**: Export cleaned data as CSV  
- **json**: Export structured data as JSON  
- **re**: Use regular expressions for precise cleaning  
- **pickle**: Save datasets for quick reload in future notebooks


In [24]:
# !pip install requests
# !pip install beautifulsoup4

In [25]:
import requests 
from bs4 import BeautifulSoup
import csv
import json
import re
import pickle

## `tdf_winner_nationalities_clean.csv`

### Dataset

We use the cleaned dataset: `tdf_winner_nationalities_clean.csv`.

This dataset provides a tidy summary of the nationalities of all Tour de France general classification winners.  
Using the “By nationality” table scraped from Wikipedia, we’ve organized, standardized, and exported:

- **Country**  
- **Number of unique Tour de France winners for that nation**  
- **List of those winning cyclists** (for reference or deeper study)

This compact dataset allows us to quickly answer:  
**Which country has produced the most unique Tour de France winners?**

It serves as a reproducible foundation for historical comparison and visual exploration.


In [26]:
URL = "https://en.wikipedia.org/wiki/List_of_Tour_de_France_general_classification_winners"

# Set headers to imitate a regular web browser
headers = {
    "User-Agent": "Mozilla/5.0"
}

# Fetch the main Wikipedia page
page = requests.get(URL, headers=headers)
soup = BeautifulSoup(page.content, "html.parser")


# Find the main content division
div = soup.find("div", {"class": "mw-content-ltr mw-parser-output"})

# Locate the table listing winners by nationality (usually 4th wikitable)
table_nationality = div.find_all("table", {"class": "wikitable"})[3]


# Extract all data rows (skip the header row)
rows = table_nationality.find_all("tr")[1:]

data = []
for row in rows:
    country = row.find("th")   # Extract country cell (from 'th')
    wins = row.find_all("td")[0]  # Extract number of wins (first 'td')
    winning_cyclist = row.find_all("td")[1]  # Extract winning cyclists (second 'td')
    # Clean and append to results
    data.append([
        country.text.strip(),
        wins.text.strip(),
        winning_cyclist.text.strip()
    ])

# Write results to a CSV file
header = ["Country", "Winners", "Winning_Cyclists"]
with open('data/tdf_winner_nationalities_clean.csv','w',encoding='utf-8',newline='') as f:
    writer = csv.writer(f) # f is the file object and we pass it to the csv.writer
    writer.writerow(header) # write the header (first row)
    writer.writerows(data) # write all data rows (data is a list of lists)

## `tdf_multiple_winners_clean.json`

### Dataset

We use the cleaned dataset `tdf_winner_nationalities_clean.csv`, which summarizes the nationalities of all Tour de France general classification winners.  
It includes the country, the number of unique winners from that nation, and the list of those cyclists.  

This compact dataset allows us to quickly answer: **Which country has produced the most unique Tour de France winners?**  
It serves as a reproducible foundation for historical comparison and visual exploration.


In [27]:
# Wikipedia URL for multiple winners section
URL = "https://en.wikipedia.org/wiki/List_of_Tour_de_France_general_classification_winners"

# Headers to mimic a browser visit
headers = {
    "User-Agent": "Mozilla/5.0"
}

# Request page content
page = requests.get(URL, headers=headers)
soup = BeautifulSoup(page.content, "html.parser")

# Find main content division
div = soup.find("div", {"class": "mw-content-ltr mw-parser-output"})

# Find the 'Multiple winners' table (3rd wikitable on the page)
table_multiple_winners = div.find_all("table", {"class": "wikitable"})[2]
rows = table_multiple_winners.find_all("tr")[1:]  # Skip header row

cyclists = []

for row in rows:
    cols = row.find_all("td")

    # Extract surname from row text with regex
    cyclist_surname = re.search(r'-value="(.+), ', str(row))
    cyclist_name = re.search(r', (\w+)"', str(row))
    country = re.search(r'<abbr title="(.+)"', str(row))

    wins_td = cols[-1]  # Last td contains years won
    years = [a.text for a in wins_td.find_all("a")]
    links_years = [f"https://en.wikipedia.org{a['href']}" for a in wins_td.find_all("a")]

    wins_years = [{"year": y, "link": l} for y, l in zip(years, links_years)]

    cyclist_data = {
        "surname": cyclist_surname.group(1),
        "name": cyclist_name.group(1),
        "country": country.group(1) if country else None,
        "win_count": len(years),
        "wins_years": wins_years
    }

    cyclists.append(cyclist_data)

# Export cleaned data as JSON
with open('data/tdf_multiple_winners_clean.json', 'w', encoding='utf-8') as f:
    json.dump(cyclists, f, indent=3)

## `tdf_shortest_tour_clean.p`

### Dataset

We use the cleaned dataset `tdf_shortest_tour_clean.p`, which contains structured information on each Tour de France edition’s length in days and kilometers.  
Each record includes the year of the Tour, the winner’s country and full name, the total distance in kilometers, and the winner’s total time or points.  

This dataset allows us to identify the shortest Tour by distance and the fastest Tour by total winning time.

In [ ]:
# URL of Wikipedia page containing Tour de France general classification winners
URL = "https://en.wikipedia.org/wiki/List_of_Tour_de_France_general_classification_winners"

# Headers to mimic a browser visit and avoid being blocked
headers = {
    "User-Agent": "Mozilla/5.0"
}

# Request page content
page = requests.get(URL, headers=headers)
soup = BeautifulSoup(page.content, "html.parser")

# Locate the main content div which holds the needed wikitable
div = soup.find("div", {"class":"mw-content-ltr mw-parser-output"})

# Select the correct wikitable (the 2nd one, index 1) with classification winners data
table_classification_winners = div.find_all("table", {"class":"wikitable"})[1]

# Extract all rows except the header row
rows = table_classification_winners.find_all("tr")[1:]

classification_data = []

for row in rows:
    cols = row.find_all("td")

    # Skip rows without a valid year link or within Lance Armstrong years (1999-2005)
    if cols[0].find("a") is None:
        continue
    year = int(cols[0].text.strip())
    # For Lance Armstrong
    if 1999 <= year <= 2005:
        continue

    # Extract needed info: year, country, cyclist name
    country = cols[1].text.strip()
    cyclist = row.find("th").text.strip()

    # Extract distance using regex and convert to float
    m_dis = re.search(r'([\d,.?]+)\s*km', str(cols[3]))
    distance_km = float(m_dis.group(1).replace(',', '')) if m_dis else None

    # Extract winning time or points
    td_time_or_points = str(cols[4])
    m_time = re.search(r'(\d+)h\s*(\d+)\′\s*(\d+)″', td_time_or_points)
    m_points = re.search(r'>(\d+)<', td_time_or_points)

    if m_time:
        # Convert time groups to total hours float
        h, m, s = map(int, m_time.groups())
        total_hours = h + m/60 + s/3600
        time_info = {
            "type": "time",
            "hours_total": round(total_hours, 2),
            "formatted": f"{h}h {m}m {s}s"
        }
    elif m_points:
        time_info = {"type": "points", "points": int(m_points.group(1))}
    else:
        time_info = {"type": "unknown", "raw": cols[4].text.strip()}

    classification_data.append({
        "year": year,
        "country": country,
        "cyclist": cyclist,
        "distance_km": distance_km,
        "time_info": time_info
    })

# Export cleaned data to pickle for later analysis
with open('data/tdf_shortest_tour_clean.p', "wb") as f:
    pickle.dump(classification_data, f)

# For visualisation
# with open('data/tdf_shortest_tour_clean.json', 'w', encoding='utf-8') as f:
#     json.dump(classification_data, f, indent=3)


## `tdf_five_timers_clean.json`

### Dataset

This dataset provides a detailed breakdown of the four cyclists who have won the Tour de France five times each, focusing on the winning times for each of their victories.  

The data is obtained by scraping the “Multiple winners” table from Wikipedia and following the links for each win year, collecting:

- Cyclist's name  
- Nationality  
- For each of their five wins:  
  - Year  
  - Wikipedia URL for that edition  
  - Winning time (in official “hh mm ss” format, as shown for first place in each year)  

All data is saved in `data/tdf_five_timers_clean.json`, an optimal format for tracking per-year performance of these unique champions.  

This structure enables time-wise comparison across race years and riders, setting up grouped bar charts or similar visualizations for analysis.

In [37]:
# Set the Wikipedia URL with the "Multiple winners" table
URL = "https://en.wikipedia.org/wiki/List_of_Tour_de_France_general_classification_winners"

# User-Agent header to mimic browser activity
headers = {"User-Agent": "Mozilla/5.0"}

# Request the main Wikipedia page content
page = requests.get(URL, headers=headers)
soup = BeautifulSoup(page.content, "html.parser")

# Find the correct table: the "Multiple winners" table (index 2)
div = soup.find("div", {"class": "mw-content-ltr mw-parser-output"})
table_multiple_winners = div.find_all("table", {"class": "wikitable"})[2]

# Select only the first 4 rows (the five-time winners)
rows = table_multiple_winners.find_all("tr")[1:5]

# Prepare the output list
cyclist_data_five_timers = []

for row in rows:
    # Extract the cyclist's name and nationality
    cyclist_name_match = re.search(r'value="([^"]+)">', str(row))
    country_match = re.search(r'<abbr title="(.+)"', str(row))

    cyclist_name = cyclist_name_match.group(1) if cyclist_name_match else ''
    country = country_match.group(1) if country_match else ''

    # Get the cell with victory years and associated links
    victories_td = row.find_all("td")[-1]
    year_links = victories_td.find_all("a")

    # Build the victories info
    victories = []
    for a in year_links:
        year = re.search(r'/wiki/(\d{4})', a['href']).group(1)
        url_year = f"https://en.wikipedia.org{a['href']}"

        # Go to that year's Tour de France page
        year_page = requests.get(url_year, headers=headers)
        year_soup = BeautifulSoup(year_page.content, "html.parser")
        content_year = year_soup.find("div", {"class": "mw-content-ltr mw-parser-output"})

        # Find the general classification table by looking for its caption
        tables = content_year.find_all("table", {"class": "wikitable"})
        table_general = None
        for t in tables:
            caption = t.find("caption")
            if caption and "Final general classification" in caption.get_text():
                table_general = t
                break

        # Extract the 1st place winning time (from second row, last column)
        if table_general:
            first_place = table_general.find_all("tr")[1]
            raw_time = first_place.find_all("td")[-1].text.strip()
            # Clean up time string
            winning_time = raw_time.replace('"', 's').strip()
        else:
            winning_time = ''

        victories.append({
            "year": year,
            "url": url_year,
            "winning_time": winning_time
        })

    cyclist_data_five_timers.append({
        "cyclist": cyclist_name,
        "country": country,
        "victories": victories
    })

# Save the structured data as JSON
with open("data/tdf_five_timers_clean.json", "w", encoding="utf-8") as f:
    json.dump(cyclist_data_five_timers, f, indent=3)


## `tdf_winner_ages_clean.csv`

### Dataset

This dataset contains clean, structured data on the age at which each Tour de France general classification winner won the event, across all years of the race.


- Scrape the main “Tour de France general classification winners” table on Wikipedia to get:
  - The year of the Tour  
  - The name of the winning cyclist  
  - The link to each cyclist’s personal Wikipedia page  

- For each winner, follow the link to their personal Wikipedia page and:
  - Scrape the date of birth (from the “infobox vcard,” typically as `<span class="bday">YYYY-MM-DD</span>`)  
  - Subtract the year of birth from the Tour year to determine the cyclist’s age at victory *(ignore months for simplicity)*

All clean data is saved as:

`data/tdf_winner_ages_clean.csv`

This CSV file provides a ready dataset for exploring patterns and trends in the age of Tour de France champions across history.

In [38]:
# URL of Wikipedia page listing Tour de France general classification winners
URL = "https://en.wikipedia.org/wiki/List_of_Tour_de_France_general_classification_winners"

# Headers to simulate a real browser visit
headers = {"User-Agent": "Mozilla/5.0"}

# Download the page content
page = requests.get(URL, headers=headers)
soup = BeautifulSoup(page.content, "html.parser")

# Find the main content division containing the data tables
div = soup.find("div", {"class": "mw-content-ltr mw-parser-output"})

# The table we want is the second table with class 'wikitable' (index 1)
table = div.find_all("table", {"class": "wikitable"})[1]

# Get all rows except the header row
rows = table.find_all("tr")[1:]

output = []

# Iterate over each table row
for row in rows:
    # Find all data cells in this row
    cols = row.find_all("td")
    # Find the header cell which contains the cyclist's name
    th = row.find("th")

    # Skip this row if:
    # - There are no columns (maybe an empty or separator row)
    # - The first column (year) lacks a link (meaning no official winner that year)
    if not cols or cols[0].find("a") is None:
        continue

    # Extract the year as string and convert to int
    year_str = cols[0].text.strip()
    year = int(year_str)

    # Exclude years 1999 to 2005 because of Lance Armstrong's disqualification
    if 1999 <= year <= 2005:
        continue

    # Extract country name from second column
    country = cols[1].text.strip()
    # Extract cyclist's full name from the header cell
    cyclist_name = th.text.strip()

    # Extract the href link to the cyclist's personal Wikipedia page
    cyclist_link = th.find("a")["href"]
    cyclist_url = f"https://en.wikipedia.org{cyclist_link}"

    # Request and parse the cyclist's personal Wikipedia page to get birthdate
    page_cyclist = requests.get(cyclist_url, headers=headers)
    soup_cyclist = BeautifulSoup(page_cyclist.content, "html.parser")
    bday_span = soup_cyclist.find("span", {"class": "bday"})

    # If birthdate found, calculate age at victory by subtracting birth year from victory year
    if bday_span:
        bday = bday_span.text
        birth_year = int(bday.split('-')[0])
        age = year - birth_year
    else:
        # Otherwise set empty values
        bday = ''
        age = ''

    # Append this record to output list
    output.append([cyclist_name, country, year, bday, age])

# Define CSV headers and write scraped data to CSV file
header = ["Cyclist", "Country", "Year", "Birthdate", "Age"]
with open("data/tdf_winner_ages_clean.csv", "w", newline='', encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(header)
    writer.writerows(output)